In [ ]:
from autogen.agentchat import AssistantAgent, UserProxyAgent
import requests

In [ ]:
# Define tool's API endpoint
API_URL =  "http://your-api-url.com/relvant" #example 

In [ ]:
# Custom Assistant Agent that uses vanna tool
class ToolAPIAgent(AssistantAgent):
    def __init__(self, name, api_url, **kwargs):
        super().__init__(name=name, **kwargs)
        self.api_url = api_url

    def generate_response(self, messages, sender, config):
        # Extract the latest user message
        user_message = messages[-1]["content"]

        # Call our API
        response = requests.post(self.api_url, json={"query": user_message})
        if response.status_code == 200:
            data = response.json()
            required_query = data.get("required_query", "No required query returned.")
            result = data.get("result", "No result returned.")

            return f"**Required Query:**\n```query\n{required_query}\n```\n\n**Result:**\n{result}"
        else:
            return f"Error calling toolagent API: {response.status_code} - {response.text}"

In [ ]:
# Instantiate agents
tool_agent = ToolAPIAgent(name="ToolAssistant", api_url=API_URL)
user_proxy = UserProxyAgent(name="User", human_input_mode="ALWAYS")

In [ ]:
# Start the conversation
chat_result = user_proxy.initiate_chat(tool_agent, message="fetch these details ...")

In [ ]:
conversation_log = []

print("Enter your question")
live_message = input()

# Start the conversation
response = user_proxy.initiate_chat(tool_agent, message=live_message, clear_history=False)
conversation_log.append({"role": "user", "content": live_message})
conversation_log.append({"role": "assistant", "content": response})

# Follow-up 1

print("Enter your question")
live_message = input()
response = user_proxy.send_message(live_message, recipient=tool_agent)
conversation_log.append({"role": "user", "content": live_message})
conversation_log.append({"role": "assistant", "content": response})

# Follow-up 2

print("Enter your question")
live_message = input()
response = user_proxy.send_message(live_message, recipient=tool_agent)
conversation_log.append({"role": "user", "content": live_message})
conversation_log.append({"role": "assistant", "content": response})


In [ ]:
"""
For user_proxy.send_message(.....)

request_reply=True (default):
The recipient agent (e.g., assistant_agent) will generate a reply immediately after receiving the message.


request_reply=False:
The message is injected silently into the conversation history.
The recipient does not reply at that moment."""